# 17 — Chat Loop

Interactive chat interface for testing the fine-tuned ARO model.
Always loads `PREFERRED_MODEL_ID` (`ARO-Lang/aro-coder-4bit`) — no fallback.

**Input:**  `config.PREFERRED_MODEL_ID` + `config.build_system_prompt()`
**Usage:**  run cells top-to-bottom, then use the chat loop cell repeatedly

## Setup

In [1]:
import sys, importlib, re, subprocess, tempfile
from pathlib import Path

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

MAX_TOKENS   = 1024
TEMPERATURE  = 0.3
TOP_P        = 0.9

print(f'Fine-tuned model: {PREFERRED_MODEL_ID}')

Fine-tuned model not found on HuggingFace, using base: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Fine-tuned model not found on HuggingFace, using base: mlx-community/Qwen3-Coder-30B-A3B-Instruct-4bit
Fine-tuned model: ARO-Lang/aro-coder-4bit


## Load model

In [2]:
load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()

print(f'Loading {PREFERRED_MODEL_ID}...')
model, tokenizer = load_fn(PREFERRED_MODEL_ID)
print('Model ready.')

kb            = load_knowledge()
SYSTEM_PROMPT = build_system_prompt(kb)

print(f'\nSystem prompt ({len(SYSTEM_PROMPT)} chars):')
print('─' * 60)
print(SYSTEM_PROMPT[:600] + ('...' if len(SYSTEM_PROMPT) > 600 else ''))
print('─' * 60)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/kris/.cache/huggingface/hub/models--ARO-Lang--aro-coder-4bit/snapshots/56ea5e93921f498795ff1394666322b75940a63e/config.json'

## Helpers

In [ ]:
def _generate(messages, max_tokens=MAX_TOKENS, temperature=TEMPERATURE):
    """Run inference on a messages list. Returns the assistant reply string."""
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    reply  = generate_fn(
        model, tokenizer,
        prompt  = tokens,
        max_tokens = max_tokens,
        sampler = make_sampler_fn(temp=temperature, top_p=TOP_P),
        verbose = False,
    )
    return reply.strip()


def extract_aro_blocks(text):
    """Return list of ARO code blocks from the response."""
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


def aro_check(code):
    """Run `aro check` on a snippet. Returns True/False/None (None = aro not found)."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, r.stdout + r.stderr
    except FileNotFoundError:
        return None, 'aro not found in PATH'
    except Exception as e:
        return None, str(e)


print('Helpers ready.')

## Chat loop

Run this cell to start an interactive session.  
Type `quit` / `exit` / `q` to stop.  
Type `reset` to clear history and start fresh.  
Type `check` to run `aro check` on any ARO blocks in the last reply.

History is preserved across iterations within one cell run — re-run to reset.

In [ ]:
history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
last_reply = ''

print('ARO Chat  (quit/exit/q to stop · reset to clear · check to validate last code)')
print('=' * 70)

while True:
    try:
        user_input = input('\nYou: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n[Interrupted]')
        break

    if not user_input:
        continue

    if user_input.lower() in ('quit', 'exit', 'q'):
        print('Goodbye.')
        break

    if user_input.lower() == 'reset':
        history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
        last_reply = ''
        print('[History cleared]')
        continue

    if user_input.lower() == 'check':
        blocks = extract_aro_blocks(last_reply)
        if not blocks:
            print('[No ARO code blocks found in last reply]')
        else:
            code = '\n\n'.join(blocks)
            ok, output = aro_check(code)
            status = '✓ PASS' if ok is True else ('✗ FAIL' if ok is False else '? (aro not in PATH)')
            print(f'[aro check: {status}]')
            if output.strip():
                print(output.strip())
        continue

    history.append({'role': 'user', 'content': user_input})

    print('\nAssistant: ', end='', flush=True)
    try:
        reply = _generate(history)
    except Exception as e:
        print(f'[Error: {e}]')
        history.pop()  # remove the user turn on failure
        continue

    print(reply)
    last_reply = reply
    history.append({'role': 'assistant', 'content': reply})

## Single-turn quick test

Useful for scripted probes without running the full interactive loop.

In [ ]:
PROMPT = 'Write a minimal ARO application that logs Hello World and exits.'

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user',   'content': PROMPT},
]

reply = _generate(messages)
print(f'PROMPT:\n{PROMPT}\n')
print(f'RESPONSE:\n{reply}\n')

blocks = extract_aro_blocks(reply)
if blocks:
    code = '\n\n'.join(blocks)
    ok, output = aro_check(code)
    status = '✓ PASS' if ok is True else ('✗ FAIL' if ok is False else '? (aro not in PATH)')
    print(f'aro check: {status}')
    if output.strip():
        print(output.strip())
else:
    print('[No ARO code blocks detected in response]')

## Batch probe

Run a fixed set of prompts and report syntax pass rate.

In [ ]:
PROBES = [
    'Write a minimal ARO Application-Start that starts an HTTP server.',
    'Write an ARO feature set that retrieves a user by ID from a repository and returns it as an OK response.',
    'How do I emit a custom event in ARO and handle it in another feature set?',
    'Write an ARO for-each loop that logs each item in a list.',
    'Explain the difference between Extract and Compute in ARO.',
    'Write an ARO feature set that validates a request body and returns a 400 status on failure.',
    'How does the Publish action work in ARO? Give an example.',
    'Write an ARO Application-Start that reads a file and logs its contents.',
]

passed = 0
no_code = 0
failed = 0

print(f'Running {len(PROBES)} probes...\n')

for i, prompt in enumerate(PROBES):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    reply  = _generate(messages)
    blocks = extract_aro_blocks(reply)

    if not blocks:
        status = '—  (no code)'
        no_code += 1
    else:
        ok, _ = aro_check('\n\n'.join(blocks))
        if ok is True:
            status = '✓  pass'
            passed += 1
        elif ok is False:
            status = '✗  FAIL'
            failed += 1
        else:
            status = '?  (aro not in PATH)'

    print(f'  [{i+1:2d}] {status:<18}  {prompt[:65]}')

total_code = passed + failed
print(f'\nSyntax pass rate: {passed}/{total_code}' + (f' ({100*passed/total_code:.0f}%)' if total_code else ''))
print(f'No-code answers:  {no_code}/{len(PROBES)}')